# Fine Tuning an LLM w/ LoRA
- Uses Python 3.12.10
- Debugging notebook

In [ ]:
from datasets import load_dataset, DatasetDict, Dataset
from transformers import AutoTokenizer, AutoConfig, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer
from peft import PeftModel, PeftConfig, get_peft_model, LoraConfig
import evaluate, torch
import numpy as np
import json


from load_custom_data import load_data_json

In [ ]:
model_checkpoint = "distilbert-base-uncased" # has 67 million parameters
# model_checkpoint = "BEE-spoke-data/smol_llama-81M-tied" # has 81 million params

# define label maps
id2label = {0: "Appropriate", 1: "Flagged", 2: "Warning"}
label2id = {"Appropriate": 0, "Warning": 1, "Flagged": 2}
# generate classification model from chechpoint
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=3, id2label=id2label, label2id=label2id)

# Load in Dataset

In [ ]:
# dataset = load_dataset("shawhin/imdb-truncated") 

dataset = load_data_json("guardian")

print(type(dataset))
print(dataset)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, add_prefix_space=True)
# add pad token if none exists
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    model.resize_token_embeddings(len(tokenizer))

In [ ]:
# create tokenize function
def tokenize_function(examples):
    # extract text
    text = examples["text"]

    #tokenize and truncate text
    tokenizer.truncation_side = "left"
    tokenized_inputs = tokenizer(
        text,
        return_tensors="np",
        truncation=True,
        max_length=512
    )

    return tokenized_inputs

In [ ]:
# tokenize training and validation datasets
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# create data collator - pads more efficiently
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(tokenized_dataset)

### Evaluating untrained model

In [ ]:
# import accuracy evaluation metric
accuracy = evaluate.load("accuracy")
# define an evaluation function to pass into trainer later
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)

    return {"accuracy": accuracy.compute(predictions=predictions, references=labels)}

In [ ]:
import json
from training.load_custom_data import load_test_data

test_list = load_test_data("guardian")

print("Untrained model predictions:")
print("----------------------------")
for text in test_list:
    # tokenize text
    inputs = tokenizer.encode(text, return_tensors="pt")
    # compute logits
    logits = model(inputs).logits
    # convert logits to label
    predictions = torch.argmax(logits)

    print(text + " - " + id2label[predictions.tolist()])

### Train model

In [ ]:
peft_config = LoraConfig(task_type="SEQ_CLS",
                        r=4,
                        lora_alpha=32,
                        lora_dropout=0.01,
                        target_modules = ['q_lin'])

print(peft_config)
model = get_peft_model(model, peft_config)
# see how many parameters we are training
model.print_trainable_parameters()

In [ ]:
# hyperparameters
lr = 1e-3
batch_size = 4
num_epochs = 10
# define training arguments
training_args = TrainingArguments(
    output_dir= model_checkpoint + "-lora-text-classification",
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)
# creater trainer object
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator, # this will dynamically pad examples in each batch to be equal length
    compute_metrics=compute_metrics,
)

# train model
trainer.train()

### Save model (and trainer)

In [ ]:
# train model (this is from your original code)
trainer.train()

# ==> Save the final, best model to a directory <==
output_directory = "./my_final_model"
trainer.save_model(output_directory)

# It's also good practice to explicitly save the tokenizer
# The tokenizer you used for training should be saved with the model
tokenizer.save_pretrained(output_directory)

### Generate Prediction

In [ ]:
model.to('cpu')

print("Trained model predictions:")
print("--------------------------")
for text in text_list:
    inputs = tokenizer.encode(text, return_tensors="pt").to("cpu")

    logits = model(inputs).logits
    predictions = torch.max(logits,1).indices

    print(text + " - " + id2label[predictions.tolist()[0]])

## Load model for deployment

In [ ]:
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer

# The path where you saved your model
model_path = "./my_final_model"

# Load the tokenizer and model from the saved directory
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

# Create a classification pipeline
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)

# Example usage
text_to_classify = "I love how easy it is to deploy Hugging Face models!"
result = classifier(text_to_classify)

print(result)
# [{'label': 'POSITIVE', 'score': 0.998}]